# IIT-08b — Parité et broadcast : le couple qui a mordu IIT

## Navigation
**Série** : [IIT](./README.md) | **Partie** : Escalier Lean (marche 2/3, lettre b)
**Précédent** : [IIT-8 : la parité, de bout en bout](./IIT-08-Parite-De-Bout-En-Bout-Lean.ipynb) | **Suivant** : [IIT-9 — le capstone](./IIT-09-Capstone-Integre-et-Trivial-Lean.ipynb)

**Marche 2 de l'escalier `iit_lean`** : après le verdict de coupe, l'outil parité et le système
broadcast. La marche 1 ([IIT-07](./IIT-07-Integration-Par-Coupes-Lean.ipynb)) a établi le vocabulaire de
la coupe et le verdict de décomposabilité : le théorème A dit que l'identité est décomposable.

Ce carnet monte les deux pièces qui manquent à la critique :

- **l'outil parité** (`parity`) et ses quatre lemmes ;
- **le système broadcast-parité** (`broadcast`), dont l'image se réduit à deux états constants ;
- **le théorème B** (`broadcast_integrated`) : le broadcast est *intégré pour toute coupe* —
  c'est « l'objet qui a mordu IIT » érigé en théorème ;
- **le théorème C** (`broadcast_two_cycle`) : la dynamique de ce même objet est *triviale* —
  toute orbite atteint un point fixe en deux pas.

Le couple **B et C** est le contenu formel de la critique d'Aaronson contre le proxy
d'intégration : intégration maximale, comportement nul.

## 0. Ce que ce carnet démontre

1. La parité d'un état se calcule (outil exécutable, mesuré sur des états concrets).
2. Le broadcast uniformise : l'état suivant ne dépend que de la parité de l'état courant.
3. Un oracle par énumération mesure les orbites : point fixe atteint en au plus deux pas,
   et la troisième itération égale toujours la deuxième (théorème C rendu visible).
4. Le théorème B atteste l'intégration universelle du même objet — le contraste avec le
   théorème A de la marche 1 est le cœur du débat.

Le carnet consomme le lake `iit_lean` (kernel `lean4-wsl`) **sans le modifier** : toutes les
définitions et théorèmes viennent de `IIT.Integration`.

In [1]:
import IIT.Integration
open IIT

-- Le contrat du carnet : outil, systeme, theoremes B et C, alias figes
#check @parity
#check @broadcast
#check @broadcast_integrated
#check @broadcast_two_cycle
#check @broadcast_trivial_dynamics

import IIT.Integration
open IIT

-- Le contrat du carnet : outil, systeme, theoremes B et C, alias figes
#check @parity
──────▶  parity : (n : Nat) → State n → Bool
#check @broadcast
──────▶  @broadcast : {n : Nat} → System n
#check @broadcast_integrated
──────▶  @broadcast_integrated : ∀ {n : Nat}, integrated broadcast
#check @broadcast_two_cycle
──────▶  @broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
#check @broadcast_trivial_dynamics
──────▶  @broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
--% env 0

Raw input:
{"cmd": "import IIT.Integration\nopen IIT\n\n-- Le contrat du carnet : outil, systeme, theoremes B et C, alias figes\n#check @parity\n#check @broadcast\n#check @broadcast_integrated\n#check @broadcast_two_cycle\n#check @broadcast_trivial_dynamics"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data": "parity : (n : Nat) → State n → Bool"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "@broadcast : {n : Nat} → System n"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data": "@broadcast_integrated : ∀ {n : Nat}, integrated broadcast"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "@broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"}],
 "env": 0}

Lecture du contrat :

- `parity : (n : Nat) → State n → Bool` — la parité (nombre de bits vrais modulo 2),
  par récurrence sur la taille ;
- `broadcast : System n` — le système : chaque bit de l'état suivant vaut la parité de
  l'état courant ;
- `broadcast_integrated : integrated broadcast` — le théorème B ;
- `broadcast_two_cycle` — le théorème C : `broadcast (broadcast (broadcast s)) = broadcast (broadcast s)` ;
- `broadcast_trivial_dynamics` — l'**énoncé figé n° 3 de #18599**, alias du théorème C sous
  le nom gelé dans l'issue fondatrice du lake.

## 1. L'outil parité

Tout le carnet repose sur un seul chiffre : la **parité** d'un état, c'est-à-dire le nombre de bits
vrais compté modulo 2. C'est le plus petit résumé possible d'un état de `n` bits — un `Bool` pour
`n` bits — et c'est exactement ce que le système de la section 2 diffusera partout.

La section installe l'outil et ses observables :

- trois **états témoins** en taille 3, choisis pour avoir des formes très différentes : un bit
  isolé au bord, tous les bits vrais, un bit isolé au milieu ;
- les fonctions `bits2` et `bits3`, qui rendent le **tuple** des bits d'un état.

Le second point mérite un mot, parce qu'il est une contrainte du langage et non un choix
pédagogique. Un état est une fonction `Fin n → Bool`. Or deux fonctions ne s'affichent pas : Lean
n'a pas d'instance d'affichage pour `Fin n → Bool`, et l'égalité décisionnelle sur les fonctions
n'existe pas non plus — `funext` est un axiome, pas une procédure de calcul. On ne peut donc
*voir* un état qu'en l'évaluant sur chacune de ses coordonnées : c'est précisément ce que fait
`bits3 s = (s 0, s 1, s 2)`. Les tuples sont le seul poste d'observation du carnet, et c'est aussi
la forme sous laquelle les tables d'orbites de la section 3 seront imprimées.

In [2]:
-- Etats temoins en taille 3
def sOneHot : State 3 := fun j => decide (j.val = 0)
def sTrue3  : State 3 := fun _ => true
def sMix3    : State 3 := fun j => decide (j.val = 1)

-- Tuples de bits (l'egalite decisionnelle vit sur les tuples, pas sur les fonctions)
def bits2 (s : State 2) : Bool × Bool := (s 0, s 1)
def bits3 (s : State 3) : Bool × Bool × Bool := (s 0, s 1, s 2)

-- Trois mesures de parite
#eval bits3 sOneHot
#eval bits3 sTrue3
#eval bits3 sMix3
#eval parity 3 sOneHot
#eval parity 3 sTrue3
#eval parity 3 sMix3

-- Etats temoins en taille 3
def sOneHot : State 3 := fun j => decide (j.val = 0)
def sTrue3  : State 3 := fun _ => true
def sMix3    : State 3 := fun j => decide (j.val = 1)

-- Tuples de bits (l'egalite decisionnelle vit sur les tuples, pas sur les fonctions)
def bits2 (s : State 2) : Bool × Bool := (s 0, s 1)
def bits3 (s : State 3) : Bool × Bool × Bool := (s 0, s 1, s 2)

-- Trois mesures de parite
#eval bits3 sOneHot
─────▶  (true, false, false)
#eval bits3 sTrue3
─────▶  (true, true, true)
#eval bits3 sMix3
─────▶  (false, true, false)
#eval parity 3 sOneHot
─────▶  true
#eval parity 3 sTrue3
─────▶  true
#eval parity 3 sMix3
─────▶  true
--% env 1

Raw input:
{"cmd": "-- Etats temoins en taille 3\ndef sOneHot : State 3 := fun j => decide (j.val = 0)\ndef sTrue3  : State 3 := fun _ => true\ndef sMix3    : State 3 := fun j => decide (j.val = 1)\n\n-- Tuples de bits (l'egalite decisionnelle vit sur les tuples, pas sur les fonctions)\ndef bits2 (s : State 2) : Bool \u00d7 Bool := (s 0, s 1)\ndef bits3 (s : State 3) : Bool \u00d7 Bool \u00d7 Bool := (s 0, s 1, s 2)\n\n-- Trois mesures de parite\n#eval bits3 sOneHot\n#eval bits3 sTrue3\n#eval bits3 sMix3\n#eval parity 3 sOneHot\n#eval parity 3 sTrue3\n#eval parity 3 sMix3", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "(true, false, false)"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 5},
   "data": "(true, true, true)"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 5},
   "data": "(false, true, false)"},
  {"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "true"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 5},
   "data": "true"},
  {"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 5},
   "data": "true"}],
 "env": 1}

Lecture : `(true, false, false)` porte un bit vrai — parité `true` ;
`(true, true, true)` en porte trois — parité `true` ; `(false, true, false)` en porte un —
parité `true`.

Les trois états témoins ont donc la même parité alors que leurs formes n'ont rien en commun : un
bit isolé au bord, le cube plein, un bit isolé au milieu. C'est la première leçon du carnet, et
elle est plus forte qu'elle n'en a l'air : la parité ne compte pas les bits vrais au sens ordinaire,
elle en prend la **somme modulo 2**. Deux états qui diffèrent d'un nombre pair de bits ont la même
image.

Autrement dit, `parity n` est une application de `State n` — soit `2^n` états — vers `Bool`, qui a
deux valeurs : elle **quotiente** l'espace des états en deux classes de `2^(n-1)` états chacune.
Tout ce qui suit en découle — le système de la section 2, les orbites de la section 3, les
théorèmes B et C. Un système qui n'observe de son entrée que cette classe ne peut pas, en sortie,
distinguer deux états de la même classe ; c'est ce qui rendra le théorème B si facile à croire et
le théorème C si facile à prouver.

In [3]:
-- Les quatre lemmes de l'outil parite
#check @parity_allFalse
#check @parity_allTrue
#check @parity_oneHot_zero
#check @parity_const

-- Les quatre lemmes de l'outil parite
#check @parity_allFalse
──────▶  parity_allFalse : ∀ (n : Nat) (s : State n), (∀ (i : Fin n), s i = false) → parity n s = false
#check @parity_allTrue
──────▶  @parity_allTrue : ∀ {n : Nat}, (parity n fun x => true) = decide (n % 2 = 1)
#check @parity_oneHot_zero
──────▶  parity_oneHot_zero : ∀ (n : Nat), 0 < n → (parity n fun j => decide (↑j = 0)) = true
#check @parity_const
──────▶  parity_const : ∀ (n : Nat) (b : Bool), (parity n fun x => b) = (b && decide (n % 2 = 1))
--% env 2

Raw input:
{"cmd": "-- Les quatre lemmes de l'outil parite\n#check @parity_allFalse\n#check @parity_allTrue\n#check @parity_oneHot_zero\n#check @parity_const", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "parity_allFalse : ∀ (n : Nat) (s : State n), (∀ (i : Fin n), s i = false) → parity n s = false"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@parity_allTrue : ∀ {n : Nat}, (parity n fun x => true) = decide (n % 2 = 1)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "parity_oneHot_zero : ∀ (n : Nat), 0 < n → (parity n fun j => decide (↑j = 0)) = true"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "parity_const : ∀ (n : Nat) (b : Bool), (parity n fun x => b) = (b && decide (n % 2 = 1))"}],
 "env": 2}

Lecture des quatre lemmes : ce ne sont pas quatre énoncés indépendants, mais une **caisse à
outils** dont la preuve du théorème B se servira, et il vaut la peine de voir comment ils
s'articulent.

- `parity_allFalse` : un état tout-faux a une parité nulle. C'est la base du calcul, et la raison
  pour laquelle l'état tout-faux sera un point fixe du broadcast en toute taille.
- `parity_allTrue` : un état tout-vrai code **l'imparité de la taille** `n`. La parité d'un état
  uniforme ne dépend pas de la valeur du bit mais du **nombre** de bits : en taille paire, un cube
  plein se comporte comme le vide.
- `parity_oneHot_zero` : un seul bit vrai, à l'indice 0, rend la parité impaire quelle que soit la
  taille. C'est le lemme pivot de la preuve du théorème B : il fournit un état dont on *connaît* la
  parité sans avoir à la calculer.
- `parity_const` : la forme générale, `b && decide (n % 2 = 1)`. Les deux premiers lemmes en sont
  les cas `b = false` et `b = true`, et `parity_allTrue` y joue le rôle du facteur de taille.

Ce dernier point est celui qui resservira le plus. Appliqué à l'état produit par `broadcast` — qui
est *toujours* un état constant, par définition —, il ramène la question « que fait le broadcast
deux fois de suite ? » à une identité sur un seul `Bool`. C'est littéralement la preuve du
théorème C, deux sections plus bas.

In [4]:
-- Mesure du lemme parity_const sur les deux tailles
#eval parity 4 (fun _ => true)
#eval parity 5 (fun _ => true)

-- Mesure du lemme parity_const sur les deux tailles
#eval parity 4 (fun _ => true)
─────▶  false
#eval parity 5 (fun _ => true)
─────▶  true
--% env 3

Raw input:
{"cmd": "-- Mesure du lemme parity_const sur les deux tailles\n#eval parity 4 (fun _ => true)\n#eval parity 5 (fun _ => true)", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "true"}],
 "env": 3}

Mesure : en taille 4 (pair), l'état tout-vrai a une parité `false` ; en taille 5 (impair),
une parité `true` — exactement `decide (n % 2 = 1)`, le comportement promis par
`parity_const`.

Ce n'est pas une coquetterie : la mesure dit deux choses sur la suite du carnet.

1. En taille **impaire**, l'état tout-vrai est sa propre image par `broadcast` — il est de parité
   `true`, donc il se diffuse vers l'état tout-vrai. C'est un **point fixe**, et le seul état
   uniforme non trivial à l'être.
2. En taille **paire**, le même état est de parité `false` : `broadcast` l'envoie sur l'état
   tout-faux. Il n'est donc jamais un point fixe, et l'oracle de la section 3 ne le verra jamais se
   stabiliser.

Autrement dit, c'est la parité de la **taille** — et non la forme des états — qui gouverne le
nombre de points fixes du système. Les tables d'orbites qui suivent le confirment sur les deux
tailles mesurables à la main : une seule cible en taille 2, deux en taille 3.

## 2. Le système broadcast : un pas

Le vocabulaire du lake est fixé ici : un **système** est une fonction d'états,
`System n := State n → State n`. L'évolution est donc déterministe et sans mémoire : l'état
suivant ne dépend que de l'état courant, et de rien d'autre.

`broadcast` est le système le plus simple qui ne soit pas l'identité. `broadcast s` est l'état
**uniforme** dont chaque bit vaut `parity n s` :

- il commence par **réduire** l'état à un seul bit, sa parité ;
- puis il **diffuse** ce bit sur les `n` éléments, sans distinction.

Ces deux gestes dans cet ordre ont une conséquence immédiate : l'image de `broadcast` ne contient
que deux états, l'état tout-faux et l'état tout-vrai, et le second n'est atteint que si la parité
est impaire. Un système dont le domaine a `2^n` points et l'image deux — c'est de là que viendront
toutes les surprises du carnet, et c'est aussi pourquoi les preuves des sections 4 et 5 tiennent en
quelques lignes.

Le contraste avec la marche 1 est à garder en tête : là, on testait l'**identité**, le système le
plus riche en comportement local, et elle se découpait en deux. Ici, on teste l'objet qui ne fait
que recopier un bit.

In [5]:
-- Un pas de broadcast sur les temoins
#eval bits3 (broadcast sOneHot)
#eval bits3 (broadcast sTrue3)
#eval bits3 (broadcast sMix3)

-- Un pas de broadcast sur les temoins
#eval bits3 (broadcast sOneHot)
─────▶  (true, true, true)
#eval bits3 (broadcast sTrue3)
─────▶  (true, true, true)
#eval bits3 (broadcast sMix3)
─────▶  (true, true, true)
--% env 4

Raw input:
{"cmd": "-- Un pas de broadcast sur les temoins\n#eval bits3 (broadcast sOneHot)\n#eval bits3 (broadcast sTrue3)\n#eval bits3 (broadcast sMix3)", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "(true, true, true)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "(true, true, true)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "(true, true, true)"}],
 "env": 4}

Lecture : les trois états témoins (tous de parité impaire) sont envoyés sur le **même**
état uniforme `(true, true, true)`. Le broadcast oublie tout de l'état courant sauf sa
parité — c'est une projection à un bit d'information, puis une diffusion.

Trois entrées distinctes, une seule sortie : `broadcast` n'est pas injectif, et il ne peut pas
l'être — son image a deux éléments quand son domaine en a `2^n`. Cette perte d'information est ce
qui rend le système intéressant pour la critique : ce qui est conservé n'est pas la forme de
l'état, mais une quantité **globale**, la même vue de partout.

Le mot « diffusé » est à prendre au sens propre. Chaque élément de l'état suivant vaut la parité de
l'état **entier**, et non celle de quelques voisins : la valeur d'un bit de sortie dépend de tous
les bits d'entrée. C'est une dépendance totale, et c'est exactement ce qu'un proxy d'intégration
cherche à récompenser. La suite du carnet montre que ce critère ne suffit pas.

## 3. Oracle : les orbites par énumération

Avant de lire les théorèmes, on mesure. L'oracle énumère les `2^n` états et imprime, pour chaque
état de départ, le tuple de bits après 0, 1, 2 et 3 applications du broadcast. C'est une
**énumération exhaustive**, pas un échantillon : sur les tailles 2 et 3 elle couvre tout l'espace
d'états, et une seule ligne qui contredirait l'énoncé du théorème C suffirait à le réfuter.

Le prix de cette exhaustivité est le `2^n` : la table est lisible à la main en taille 2 (4 lignes)
et en taille 3 (8 lignes), pénible en taille 8 (256 lignes). C'est la raison d'être des théorèmes —
la mesure convainc sur les petites tailles, la preuve couvre toutes les tailles d'un coup. Un
carnet qui n'aurait que l'oracle aurait un angle mort : les tailles qu'on n'a pas énumérées.

L'énumération passe par un **encodage** de l'état en entier naturel, qu'il faut lire avant les
tables. C'est l'objet de la cellule suivante.

In [6]:
-- Oracle : enumeration exhaustive des etats par leur code entier
def stateOfNat (n : Nat) (v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1
def allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)

#eval (allStates 2).map bits2

-- Oracle : enumeration exhaustive des etats par leur code entier
def stateOfNat (n : Nat) (v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1
def allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)

#eval (allStates 2).map bits2
─────▶  [(false, false), (true, false), (false, true), (true, true)]
--% env 5

Raw input:
{"cmd": "-- Oracle : enumeration exhaustive des etats par leur code entier\ndef stateOfNat (n : Nat) (v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1\ndef allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)\n\n#eval (allStates 2).map bits2", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "[(false, false), (true, false), (false, true), (true, true)]"}],
 "env": 5}

Lecture de l'encodeur, avant de lire les tables.

`stateOfNat n v` construit un état à partir d'un **code entier** `v` : le bit d'indice `i` vaut
`true` si et seulement si le `i`-ième bit binaire de `v` est à 1, c'est-à-dire si
`(v >>> i) % 2 == 1`. L'indice de l'élément est donc la **position du bit** dans l'écriture
binaire, en partant des unités. En taille 2, les quatre codes rendent :

| code `v` | écriture binaire | état |
|---|---|---|
| 0 | `00` | `(false, false)` |
| 1 | `01` | `(true, false)` |
| 2 | `10` | `(false, true)` |
| 3 | `11` | `(true, true)` |

`allStates n` prend ensuite `List.range (2^n)` — les codes dans l'ordre croissant — et applique
l'encodeur. L'ordre d'impression de la sortie ci-dessus est donc l'ordre des **codes**, pas l'ordre
lexicographique des tuples : `(true, false)` précède `(false, true)` parce que `1 < 2` et que le
premier élément de la paire est le bit de poids faible. Deux conséquences pratiques pour la lecture
des tables qui suivent :

- un état et son état **transposé**, celui dont les bits sont échangés, occupent des lignes voisines
  mais distinctes : leur ressemblance n'est pas une redondance ;
- l'ordre des lignes n'a aucune signification dynamique. Ce n'est pas l'ordre des orbites mais
  l'ordre d'énumération ; une orbite se lit **horizontalement**, d'une colonne à la suivante.

L'encodage est aussi une bijection, et la sortie le montre : les quatre états imprimés sont deux à
deux distincts, et il y en a exactement `2^2`. Si l'encodeur écrasait deux codes sur le même état,
la table d'orbites passerait silencieusement à côté d'un état de départ — défaut qu'une énumération
ne signale pas d'elle-même, et que le compte rend visible.

Une remarque de forme, pour ne pas s'arrêter sur un détail d'affichage : dans les tables d'orbites,
les deux dernières entrées s'impriment sans parenthèses — `..., false, false` au lieu de
`..., (false, false)`. Les paires de `Bool × Bool` sont imbriquées dans un n-uplet, et l'affichage
de Lean n'entoure de parenthèses que certaines composantes. Il faut donc lire ces deux dernières
entrées comme les deux bits d'une **seule** paire, et non comme deux valeurs séparées.

In [7]:
-- Orbites en taille 2 : (depart, pas 1, pas 2, pas 3)
#eval (allStates 2).map (fun s =>
  (bits2 s,
   bits2 (broadcast s),
   bits2 (broadcast (broadcast s)),
   bits2 (broadcast (broadcast (broadcast s)))))

-- Orbites en taille 2 : (depart, pas 1, pas 2, pas 3)
#eval (allStates 2).map (fun s =>
─────▶  [((false, false), (false, false), (false, false), false, false),
 ((true, false), (true, true), (false, false), false, false),
 ((false, true), (true, true), (false, false), false, false),
 ((true, true), (false, false), (false, false), false, false)]
  (bits2 s,
   bits2 (broadcast s),
   bits2 (broadcast (broadcast s)),
   bits2 (broadcast (broadcast (broadcast s)))))
--% env 6

Raw input:
{"cmd": "-- Orbites en taille 2 : (depart, pas 1, pas 2, pas 3)\n#eval (allStates 2).map (fun s =>\n  (bits2 s,\n   bits2 (broadcast s),\n   bits2 (broadcast (broadcast s)),\n   bits2 (broadcast (broadcast (broadcast s)))))", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "[((false, false), (false, false), (false, false), false, false),\n ((true, false), (true, true), (false, false), false, false),\n ((false, true), (true, true), (false, false), false, false),\n ((true, true), (false, false), (false, false), false, false)]"}],
 "env": 6}

Lecture de la table (taille 2, paire) :

- chaque ligne converge vers `(false, false)` — l'unique point fixe en taille paire
  (l'état tout-vrai a une parité nulle, cf. la mesure de `parity_const` en taille 4) ;
- `(false, true)` et `(true, false)` passent par `(true, true)` au pas 1 puis atteignent
  leur point fixe au pas 2 : convergence en **au plus deux pas** ;
- **les colonnes 3 et 4 sont identiques partout** : la troisième itération ne change rien —
  le théorème C est déjà visible dans la mesure.

La table se lit mieux à l'envers : en partant de la **sortie**, il n'y a que deux états possibles,
tout-faux et tout-vrai, et une seule des deux est un point fixe. Les quatre départs se répartissent
donc en trois familles :

| départ | parité | pas 1 | pas 2 | pas 3 |
|---|---|---|---|---|
| `(false, false)` | paire | `(false, false)` | `(false, false)` | `(false, false)` |
| `(true, false)` | impaire | `(true, true)` | `(false, false)` | `(false, false)` |
| `(false, true)` | impaire | `(true, true)` | `(false, false)` | `(false, false)` |
| `(true, true)` | paire | `(false, false)` | `(false, false)` | `(false, false)` |

Ce qu'il faut retenir n'est pas la convergence — elle est acquise dès qu'on sait que l'image a deux
points — mais ceci : la colonne du pas 1 ne dépend que de la **parité de la ligne**, et les lignes
de même parité deviennent indistinguables dès le premier pas. Les deux états impairs sont écrasés
l'un sur l'autre, les deux états pairs aussi. La partition en deux classes annoncée en section 1
devient ici une partition en deux **destins**.

Ce que la table ne montre pas, et qu'aucune table ne montrera : le comportement en taille 4, 6, 100.
Elle établit le motif sur un cas, le théorème C le transporte à toutes les tailles. C'est
l'articulation à garder en tête dans tout le carnet : la mesure instruit, la preuve généralise.

In [8]:
-- Orbites en taille 3 : (depart, pas 1, pas 2, pas 3)
#eval (allStates 3).map (fun s =>
  (bits3 s,
   bits3 (broadcast s),
   bits3 (broadcast (broadcast s)),
   bits3 (broadcast (broadcast (broadcast s)))))

-- Orbites en taille 3 : (depart, pas 1, pas 2, pas 3)
#eval (allStates 3).map (fun s =>
─────▶  [((false, false, false), (false, false, false), (false, false, false), false, false, false),
 ((true, false, false), (true, true, true), (true, true, true), true, true, true),
 ((false, true, false), (true, true, true), (true, true, true), true, true, true),
 ((true, true, false), (false, false, false), (false, false, false), false, false, false),
 ((false, false, true), (true, true, true), (true, true, true), true, true, true),
 ((true, false, true), (false, false, false), (false, false, false), false, false, false),
 ((false, true, true), (false, false, false), (false, false, false), false, false, false),
 ((true, true, true), (true, true, true), (true, true, true), true, true, true)]
  (bits3 s,
   bits3 (broadcast s),
   bits3 (broadcast (broadcast s)),
   bits3 (broadcast (broadcast (broadcast s)))))
--% env 7

Raw input:
{"cmd": "-- Orbites en taille 3 : (depart, pas 1, pas 2, pas 3)\n#eval (allStates 3).map (fun s =>\n  (bits3 s,\n   bits3 (broadcast s),\n   bits3 (broadcast (broadcast s)),\n   bits3 (broadcast (broadcast (broadcast s)))))", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "[((false, false, false), (false, false, false), (false, false, false), false, false, false),\n ((true, false, false), (true, true, true), (true, true, true), true, true, true),\n ((false, true, false), (true, true, true), (true, true, true), true, true, true),\n ((true, true, false), (false, false, false), (false, false, false), false, false, false),\n ((false, false, true), (true, true, true), (true, true, true), true, true, true),\n ((true, false, true), (false, false, false), (false, false, false), false, false, false),\n ((false, true, true), (false, false, false), (false, false, false), false, false, false),\n ((true, true, true), (true, true, true), (true, true, true), true, true, true)]"}],
 "env": 7}

Lecture de la table (taille 3, impaire) : **deux** points fixes cette fois.

- Les états de parité paire (0 ou 2 bits vrais) convergent vers `(false, false, false)`.
- Les états de parité impaire (1 ou 3 bits vrais) convergent vers `(true, true, true)` —
  qui est lui-même un point fixe : en taille impaire, la parité de l'état tout-vrai vaut
  `true`, l'état se diffuse à lui-même.
- Colonnes 3 = 4 partout, convergence en au plus deux pas : le théorème C tient pour
  les deux tailles.

Les huit lignes se partagent en deux bassins de **quatre** états chacun, un partage exact et non un
déséquilibre. Le compte se vérifie sans énumérer : en taille 3 il y a `C(3,0) + C(3,2) = 1 + 3`
états de parité paire et `C(3,1) + C(3,3) = 3 + 1` états de parité impaire. Le système ne distingue
que la classe de parité, donc les bassins ont la taille des classes, et les coefficients binomiaux
disent que ces classes sont équilibrées.

En taille 2, le même raisonnement donnait `C(2,0) + C(2,2) = 2` états pairs et `C(2,1) = 2` états
impairs — mais le bassin impair a disparu, parce que sa cible `(true, true)` n'est pas un point fixe
en taille paire. Le nombre de points fixes n'est donc pas une propriété du système seul : il dépend
de la parité de la taille, ce que `parity_const` encode précisément.

C'est la seconde leçon du carnet, et elle est contre-intuitive : **doubler l'information ne fait pas
doubler la diversité**. On passe de 4 à 8 états de départ, et le nombre d'attracteurs passe de 1 à
2, pas de 1 à 4. La dynamique reste gouvernée par un seul bit.

## 4. Théorème C — la dynamique est triviale

Le théorème C est l'énoncé qui transforme la colonne 3 de l'oracle en fait général :

```lean
∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
```

Trois applications valent deux applications, pour **tout** état et **toute** taille. En termes
d'algèbre, la composée `broadcast ∘ broadcast` est **idempotente** : la réappliquer ne change plus
rien. En termes de dynamique, toute orbite est stabilisée au bout de deux pas — un point fixe est
atteint, et il l'est vite.

L'énoncé est plus fort qu'il n'y paraît, et il faut voir pourquoi il ne pouvait pas être
`broadcast (broadcast s) = broadcast s`. Cette version-là serait **fausse** : en taille 2, l'état
`(true, false)` a pour image `(true, true)`, qui n'est pas lui-même. Le broadcast n'est pas
idempotent ; c'est seulement sa **seconde** itération qui l'est. L'écart entre les deux énoncés est
exactement la perte d'information : la première application réduit un état à sa classe de parité, la
seconde ne fait que recopier une valeur déjà constante.

La cellule suivante interroge le noyau sur deux choses : la signature de l'énoncé, puis ses
**dépendances axiomatiques**. La seconde compte autant que la première — « c'est un théorème » et
« c'est un théorème qui ne suppose rien d'autre que le langage » sont deux affirmations
différentes, et seule la seconde est une garantie.

In [9]:
-- Theoreme C et son alias figé
#check @broadcast_two_cycle
#print axioms broadcast_two_cycle
#check @broadcast_trivial_dynamics
#print axioms broadcast_trivial_dynamics

-- Theoreme C et son alias figé
#check @broadcast_two_cycle
──────▶  @broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
#print axioms broadcast_two_cycle
──────▶  'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]
#check @broadcast_trivial_dynamics
──────▶  @broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
#print axioms broadcast_trivial_dynamics
──────▶  'IIT.broadcast_trivial_dynamics' depends on axioms: [propext, Quot.sound]
--% env 8

Raw input:
{"cmd": "-- Theoreme C et son alias fig\u00e9\n#check @broadcast_two_cycle\n#print axioms broadcast_two_cycle\n#check @broadcast_trivial_dynamics\n#print axioms broadcast_trivial_dynamics", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "'IIT.broadcast_trivial_dynamics' depends on axioms: [propext, Quot.sound]"}],
 "env": 8}

L'énoncé : `broadcast (broadcast (broadcast s)) = broadcast (broadcast s)` pour tout état
`s`. Toute orbite atteint un point fixe en deux pas — la **dynamique est triviale** :
l'information globale est préservée (la parité), mais il ne se « passe » plus rien.
L'alias `broadcast_trivial_dynamics` porte le nom figé par l'issue #18599 : le lake garantit
cet énoncé sous ce nom, quelle que soit l'évolution des preuves internes.

La preuve, telle qu'elle est écrite dans le lake, tient en trois gestes, et les nommer aide à
comprendre l'énoncé mieux qu'une paraphrase :

1. `funext` ramène l'égalité de fonctions à une égalité point par point, puis la définition de
   `broadcast` remplace chaque membre par une parité d'état **constant** ;
2. `parity_const` est appliqué deux fois, ce qui réduit chaque membre à une expression sur un seul
   `Bool`, où figure des deux côtés le facteur de taille `decide (n % 2 = 1)` ;
3. il ne reste qu'une identité booléenne, de la forme `(b && c) && c = b && c`, refermée par
   `simp`.

Autrement dit, tout le contenu du théorème C est l'**idempotence de `&&`**. C'est ce qui explique
qu'il vaille pour toute taille et tout état sans aucune hypothèse : le broadcast jette l'information
de forme, et ce qui reste — un bit — ne peut pas être re-jeté.

La conséquence sur les points fixes se lit directement sur `parity_const`. Un point fixe du
broadcast est nécessairement un état **constant**, puisque l'image de `broadcast` n'est faite que
d'états constants ; et un état constant `fun _ => v` est un point fixe si et seulement si
`v && decide (n % 2 = 1) = v`. Le cas `v = false` l'est toujours ; le cas `v = true` ne l'est que
si `n` est impair. Il y a donc **un ou deux** points fixes selon la parité de la taille, et jamais
davantage — ce que les tables de la section 3 montraient déjà, et ce qu'un lecteur peut vérifier sur
`n = 4` et `n = 5` sans énumérer 16 et 32 états.

In [10]:
-- Temoin concret : le theorique C mesure sur un etat
#eval bits3 (broadcast (broadcast (broadcast sOneHot))) == bits3 (broadcast (broadcast sOneHot))

-- Temoin concret : le theorique C mesure sur un etat
#eval bits3 (broadcast (broadcast (broadcast sOneHot))) == bits3 (broadcast (broadcast sOneHot))
─────▶  true
--% env 9

Raw input:
{"cmd": "-- Temoin concret : le theorique C mesure sur un etat\n#eval bits3 (broadcast (broadcast (broadcast sOneHot))) == bits3 (broadcast (broadcast sOneHot))", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "true"}],
 "env": 9}

Le témoin rend `true` : sur l'état témoin, la troisième itération est byte-identique à la
deuxième. La mesure de l'oracle (colonnes 3 = 4) et le théorème C disent la même chose —
l'un pour `2^n` états mesurés, l'autre pour tous les états de toutes les tailles.

La distinction vaut d'être posée précisément, parce que c'est la frontière épistémique du carnet :

- la cellule ci-dessus est un `#eval` : Lean **calcule** l'égalité sur un état nommé et rend
  `true`. C'est la preuve qu'un témoin existe, pas une preuve universelle. Un seul état
  contradictoire aurait suffi à la réfuter, mais aucun nombre d'états concordants ne la démontre ;
- le théorème C est une **preuve** : quantifiée sur tous les `n` et tous les `s`, elle ne dépend
  d'aucun calcul, et `#print axioms` atteste qu'elle ne repose sur rien d'autre que `propext` et
  `Quot.sound`.

Le carnet livre les deux, et c'est délibéré : la mesure est ce qui rend la preuve croyable, la
preuve est ce qui rend la mesure générale. Un cours qui ne montrerait que la mesure laisserait
croire que « ça se stabilise » est une loi approchée ; un cours qui ne montrerait que la preuve
laisserait le lecteur devant un énoncé sans intuition.

## 5. Théorème B — le broadcast est intégré pour toute coupe

Le théorème B est l'autre moitié du couple, et il demande d'abord de fixer ce que « intégré » veut
dire. Le lake donne la définition en trois étages, et l'ordre compte.

Une **coupe** d'un système à `n` éléments est un partage des indices en deux côtés : `c.sel i` vaut
`true` si l'élément `i` est côté A, `false` s'il est côté B. La coupe est **non triviale** quand les
deux côtés sont habités, ce qu'attestent les champs `c.ha` et `c.hb` — deux existences, une de
chaque côté.

Un **motif** de sortie est ce qu'une moitié du système produit : `projT c` garde les bits côté A et
met les autres à `false`, `projF c` fait le symétrique. Une coupe est **indépendante** pour `f`
quand tout couple de motifs réalisables *séparément* est réalisable *ensemble* : pour tous `x` et
`y`, il existe un état `z` dont le motif A égale celui de `x` et le motif B celui de `y`. C'est la
définition exacte de « les deux moitiés vivent leur vie » — l'image du système est le produit des
images de ses parties.

Un système est alors **décomposable** s'il admet une coupe indépendante, et **intégré** dans le cas
contraire : aucune coupe ne le sépare en deux moitiés autonomes. C'est le proxy d'intégration tel
que la littérature le discute, formalisé à sa racine : non pas la mesure `Φ` de la théorie, mais le
critère combinatoire dont elle s'inspire.

Reste une asymétrie de quantification à ne pas manquer, parce qu'elle fait toute la force du
théorème B : la décomposition est une **existence** — une coupe suffit —, l'intégration est un
**universel** — il faut les réfuter toutes. Prouver `integrated f`, c'est donc prouver une négation
sur un objet infini, par contradiction. C'est exactement la forme de la preuve ci-dessous.

In [11]:
-- Theoreme B : l'objet qui a mordu IIT, en theoreme
#check @broadcast_integrated
#print axioms broadcast_integrated
#check @integrated

-- Theoreme B : l'objet qui a mordu IIT, en theoreme
#check @broadcast_integrated
──────▶  @broadcast_integrated : ∀ {n : Nat}, integrated broadcast
#print axioms broadcast_integrated
──────▶  'IIT.broadcast_integrated' depends on axioms: [propext]
#check @integrated
──────▶  @integrated : {n : Nat} → System n → Prop
--% env 10

Raw input:
{"cmd": "-- Theoreme B : l'objet qui a mordu IIT, en theoreme\n#check @broadcast_integrated\n#print axioms broadcast_integrated\n#check @integrated", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "@broadcast_integrated : ∀ {n : Nat}, integrated broadcast"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "'IIT.broadcast_integrated' depends on axioms: [propext]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data": "@integrated : {n : Nat} → System n → Prop"}],
 "env": 10}

Lecture du théorème B : pour **toute coupe non triviale** (`c.ha` et `c.hb` attestent que
les deux côtés existent), le couple de motifs — côté A diffusé *vrai* **et** côté B diffusé
*faux* — n'est **jamais réalisé ensemble**, alors que chaque motif l'est séparément.

Autrement dit : aucune coupe ne découpe le broadcast en deux moitiés indépendantes. Le
système est **intégré au sens du proxy, universellement** — c'est exactement « l'objet qui a
mordu IIT » (l'exemple qui a fait réaliser à la communauté que le proxy d'intégration
récompensait cet objet), désormais un théorème du lake.

La preuve mérite d'être suivie pas à pas sur un cas concret, parce qu'elle est courte et que son
mécanisme est instructif. Prenons `n = 2` et la coupe qui met l'élément 0 côté A et l'élément 1 côté
B, soit `sel = (true, false)`. Raisonnons par l'absurde, en supposant la coupe indépendante.

1. On choisit `x` de parité **impaire** — par exemple l'état à un seul bit vrai, `(true, false)` —
   et `y` de parité **paire** — l'état tout-faux.
2. L'indépendance fournit un état `z` tel que `projT (broadcast z) = projT (broadcast x)` et
   `projF (broadcast z) = projF (broadcast y)`.
3. Or `broadcast z` est **uniforme** : tous ses bits valent `parity z`. Le côté A de
   `broadcast x` vaut `parity x`, soit `true` ; donc le côté A de `broadcast z` vaut `true`, donc
   `parity z = true`.
4. Le côté B de `broadcast y` vaut `parity y`, soit `false` ; donc le côté B de `broadcast z` vaut
   `false`, donc `parity z = false`.
5. Contradiction sur un seul `Bool`.

Le point décisif est aux étapes 3 et 4 : les deux côtés de la coupe **lisent la même variable**, la
parité de `z`. Un système uniforme n'a qu'un bit d'information à répartir, et ce bit est partagé par
construction — on ne peut pas le donner vrai à gauche et faux à droite. C'est en ce sens très précis
que le broadcast est intégré : non parce qu'il serait complexe, mais parce que son unique variable
est globale.

Deux détails de la preuve valent d'être signalés, parce qu'ils disent pourquoi elle tient en
quelques lignes :

- `x` n'a pas besoin d'appartenir au côté A. `broadcast` rend tout uniforme, donc chaque bit de
  sortie porte déjà la parité entière : il suffit que le côté A soit habité, ce que `c.ha` assure ;
- les deux lemmes de la section 1 font tout le travail de calcul : `parity_oneHot_zero` fournit un
  état de parité connue (`true`), `parity_allFalse` un état de parité connue (`false`). Le reste
  n'est qu'application de la définition, sans combinatoire ni cas sur `n`.

In [12]:
-- Rappel marche 1 : le contraste A vs B
#check @cutIndep_of_sides_independent
#check @product_decomposable

-- Rappel marche 1 : le contraste A vs B
#check @cutIndep_of_sides_independent
──────▶  @cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),
  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →
    (∀ (s s' : State n),
        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →
      cutIndep c f
#check @product_decomposable
──────▶  @product_decomposable : ∀ {n : Nat} (c : Cut n) (f : System n),
  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →
    (∀ (s s' : State n),
        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →
      decomposable f
--% env 11

Raw input:
{"cmd": "-- Rappel marche 1 : le contraste A vs B\n#check @cutIndep_of_sides_independent\n#check @product_decomposable", "env": 10}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),\n  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →\n    (∀ (s s' : State n),\n        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →\n      cutIndep c f"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@product_decomposable : ∀ {n : Nat} (c : Cut n) (f : System n),\n  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →\n    (∀ (s s' : State n),\n        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →\n      decomposable f"}],
 "env": 11}

Le contraste des deux théorèmes, dans le même vocabulaire :

| | système | verdict |
|---|---|---|
| **Théorème A** (`cutIndep_of_sides_independent`) | l'identité | **décomposable** : chaque moitié vit sa vie |
| **Théorème B** (`broadcast_integrated`) | le broadcast | **intégré pour toute coupe** : aucune moitié ne vit sa vie |

L'identité — le système le plus « riche » en comportement local — est décomposable ; le
broadcast — qui ne fait que diffuser un bit — est universellement intégré. Le proxy inverse
l'intuition.

L'inversion vaut d'être nommée précisément, parce que c'est elle que la critique attaque. Ce que le
proxy mesure, au fond, c'est la **non-séparabilité de l'image** : un système est décomposable quand
on peut reconstituer n'importe quelle combinaison de motifs de sortie à partir des deux moitiés
prises séparément. L'identité a une image maximale — tous les états sont atteignables — et cette
image est un produit, donc elle se sépare. Le broadcast a une image de deux points, et ces deux
points ne se séparent pas.

Autrement dit, le critère récompense une propriété de l'**image** d'un système, pas de son
**comportement** dans le temps. Un système peut donc maximiser le critère en ne faisant presque
rien : il suffit que son peu d'information soit partagé. C'est la fissure que la section suivante
referme avec le théorème C.

## 6. La critique d'Aaronson : B et C ensemble

Le couple est la critique :

- **B** : le broadcast est intégré pour toute coupe — information globalement liée ;
- **C** : sa dynamique est triviale — point fixe en deux pas, plus rien ne se passe.

Si un proxy d'intégration attache une valeur maximale à un objet dont le comportement se
réduit à « stabiliser la parité », il ne mesure pas le *calcul* — il mesure un artefact de
la définition. C'est le contenu formel de la critique d'Aaronson contre les théories
de l'information intégrée, et c'est la raison pour laquelle le lake livre les deux
théorèmes **sous le même toit** : l'un sans l'autre ne prouve rien.

Ce qu'il faut ajouter, c'est **pourquoi** C renverse B, et le renversement tient à une observation
précise : le théorème C ne dit pas seulement « ça se stabilise », il dit « ça se stabilise en deux
pas, quelle que soit la taille ». Aucune des deux précisions n'est anecdotique.

- **Deux pas, et pas plus** : une dynamique intéressante peut converger lentement, osciller,
  explorer un espace. Ici, deux itérations suffisent quelle que soit la taille, parce que la
  composée `broadcast ∘ broadcast` est déjà constante — et une fonction constante n'a pas de
  dynamique.
- **Quelle que soit la taille** : la trivialité ne s'améliore ni ne s'aggrave quand on ajoute des
  éléments. On pourrait espérer qu'un système plus grand calcule davantage ; il ne calcule pas
  plus, il diffuse le même bit plus largement.

Le désaccord entre le proxy et l'intuition porte donc sur l'**objet mesuré**, et non sur un seuil.
Le proxy demande : l'image du système se sépare-t-elle ? Le broadcast répond non, avec la plus
grande marge possible. L'intuition demande : le système calcule-t-il quelque chose ? Le broadcast
répond non, avec la plus grande marge possible. Les deux réponses sont exactes, et c'est leur
conjonction qui condamne le proxy comme mesure de la conscience : un système peut être maximalement
intégré et maximalement inerte à la fois.

C'est pourquoi le lake livre les deux théorèmes ensemble. Un lake qui n'aurait que B aurait l'air de
confirmer la théorie ; un lake qui n'aurait que C aurait l'air de décrire une bizarrerie. Le couple
est l'argument.

In [13]:
-- Les deux theoremes ne dependent que des axiomes standard
#print axioms broadcast_integrated
#print axioms broadcast_two_cycle

-- Les deux theoremes ne dependent que des axiomes standard
#print axioms broadcast_integrated
──────▶  'IIT.broadcast_integrated' depends on axioms: [propext]
#print axioms broadcast_two_cycle
──────▶  'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]
--% env 12

Raw input:
{"cmd": "-- Les deux theoremes ne dependent que des axiomes standard\n#print axioms broadcast_integrated\n#print axioms broadcast_two_cycle", "env": 11}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "'IIT.broadcast_integrated' depends on axioms: [propext]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]"}],
 "env": 12}

Mesure des dépendances : le théorème B ne dépend que de `propext` — même pas de
l'extensionnalité des fonctions ; le théorème C ajoute `Quot.sound` (qui porte `funext`,
utilisée par sa conclusion point à point). Aucune des deux preuves ne repose sur un axiome
ajouté ni sur un `sorry` : le couple B et C est aussi fort que le langage le permet.

Il faut être exact sur ce que cette mesure garantit, et sur ce qu'elle ne garantit pas.

- **Ce qu'elle garantit** : la liste est celle des axiomes que le **noyau** doit supposer pour
  accepter les preuves. Ni `sorryAx` — un trou déguisé en preuve — ni axiome déclaré à la main n'y
  figurent. Les deux axiomes présents sont ceux que la bibliothèque standard utilise partout, et ils
  sont consistants avec les fondations usuelles.
- **Ce qu'elle ne garantit pas** : que l'**énoncé** formalise bien l'intuition visée. `integrated`
  est la définition du lake, pas celle de la théorie de l'information intégrée ; la preuve dit que
  le broadcast satisfait *cette* définition. C'est tout l'argument d'Aaronson — mais un lecteur qui
  contesterait la définition ne serait pas réfuté par la preuve.

La distinction entre ces deux colonnes sépare « formellement vérifié » de « vrai », et un carnet de
vérification formelle rend service en la rappelant plutôt qu'en la masquant.

### Ce que le couple ne dit pas

Un argument formel se juge autant sur ce qu'il laisse ouvert que sur ce qu'il ferme, et trois
limites de celui-ci méritent d'être posées avant de passer aux exercices.

**1. Le broadcast est un adversaire choisi.** C'est un système dont l'image a deux points, et dont
la sortie ne dépend que d'un seul bit d'entrée. Un défenseur de la théorie répondra que réfuter un
proxy sur un système dégénéré ne dit rien des systèmes où le proxy est censé servir. La réponse est
que l'argument n'a pas besoin d'être statistique : il suffit d'un contre-exemple pour montrer que le
critère ne mesure pas ce qu'il prétend mesurer, et le broadcast en est un — reproductible, minimal,
et vérifié.

**2. La critique attaque le proxy, pas la théorie.** Le lake formalise `integrated` et le critère de
coupe ; il ne formalise pas la théorie de l'information intégrée elle-même, ni ses versions
ultérieures. Un lecteur qui conclurait « IIT est réfutée » irait plus loin que les théorèmes. Ce que
le couple établit est plus étroit et plus solide : *ce critère-là* récompense un système inerte.

**3. L'intégration et la conscience ne sont pas le même objet.** Rien dans le carnet ne parle de
conscience, d'expérience ni de sujet. Il parle d'un système dynamique fini et d'un critère
combinatoire. La critique d'Aaronson tire sa force de ce que la théorie *identifie* les deux — et
cette identification est une thèse philosophique, pas un énoncé du lake.

Ces trois limites ne diminuent pas le résultat : elles disent ce qu'il vaut. Un carnet de
vérification formelle rend service en bornant son propre énoncé plutôt qu'en le laissant enfler.

## 7. Exercices

Trois exercices, du calcul à la prédiction d'orbite. Les cellules s'exécutent telles
quelles (rien à casser) : compléter le TODO puis re-exécuter.

Chacun porte sur une compétence différente, et l'ordre n'est pas arbitraire :

1. **calculer** — prédire une parité de tête, en comptant les bits vrais, sans appeler `parity` ;
2. **déduire** — appliquer `parity_const` à une taille donnée pour trancher un cas de point fixe ;
3. **reconstituer** — décrire une orbite complète, puis vérifier contre la table de la section 3.

Les cellules rendent `"Exercice N a completer"` : c'est le contrat du stub, et il est volontaire.
Une cellule d'exercice doit s'exécuter sans erreur avant que l'étudiant y touche, sinon le carnet ne
se distingue plus d'un carnet cassé. La ligne de contrôle est déjà écrite, en commentaire : c'est
elle qu'on décommente après avoir rédigé sa prédiction, et l'ordre a son importance — prédire
d'abord, exécuter ensuite, sinon la mesure fait le raisonnement à la place du lecteur.

In [14]:
-- Exercice 1 : la parite a la main
-- TODO etudiant : predire (sans #eval parity) la parite de l'etat ci-dessous,
-- en comptant les bits vrais, puis verifier en decommentant la ligne de controle.
-- Etape 1 : ecrire la valeur attendue (true ou false) en commentaire
-- Etape 2 : decommenter et executer la ligne de controle
def sExo1 : State 3 := fun j => decide (j.val = 2)
#eval "Exercice 1 a completer"
-- #eval parity 3 sExo1

-- Exercice 1 : la parite a la main
-- TODO etudiant : predire (sans #eval parity) la parite de l'etat ci-dessous,
-- en comptant les bits vrais, puis verifier en decommentant la ligne de controle.
-- Etape 1 : ecrire la valeur attendue (true ou false) en commentaire
-- Etape 2 : decommenter et executer la ligne de controle
def sExo1 : State 3 := fun j => decide (j.val = 2)
#eval "Exercice 1 a completer"
─────▶  "Exercice 1 a completer"
-- #eval parity 3 sExo1
--% env 13

Raw input:
{"cmd": "-- Exercice 1 : la parite a la main\n-- TODO etudiant : predire (sans #eval parity) la parite de l'etat ci-dessous,\n-- en comptant les bits vrais, puis verifier en decommentant la ligne de controle.\n-- Etape 1 : ecrire la valeur attendue (true ou false) en commentaire\n-- Etape 2 : decommenter et executer la ligne de controle\ndef sExo1 : State 3 := fun j => decide (j.val = 2)\n#eval \"Exercice 1 a completer\"\n-- #eval parity 3 sExo1", "env": 12}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "\"Exercice 1 a completer\""}],
 "env": 13}

In [15]:
-- Exercice 2 : la taille gouverne le point fixe
-- TODO etudiant : d'apres le lemme parity_const (et la mesure en taille 4 et 5),
-- predire si l'etat tout-vrai de taille 6 est un point fixe du broadcast.
-- Repondre en commentaire : point fixe ou non, et pourquoi (le role de 6 % 2).
#eval "Exercice 2 a completer"


-- Exercice 2 : la taille gouverne le point fixe
-- TODO etudiant : d'apres le lemme parity_const (et la mesure en taille 4 et 5),
-- predire si l'etat tout-vrai de taille 6 est un point fixe du broadcast.
-- Repondre en commentaire : point fixe ou non, et pourquoi (le role de 6 % 2).
#eval "Exercice 2 a completer"
─────▶  "Exercice 2 a completer"

--% env 14

Raw input:
{"cmd": "-- Exercice 2 : la taille gouverne le point fixe\n-- TODO etudiant : d'apres le lemme parity_const (et la mesure en taille 4 et 5),\n-- predire si l'etat tout-vrai de taille 6 est un point fixe du broadcast.\n-- Repondre en commentaire : point fixe ou non, et pourquoi (le role de 6 % 2).\n#eval \"Exercice 2 a completer\"\n", "env": 13}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "\"Exercice 2 a completer\""}],
 "env": 14}

In [16]:
-- Exercice 3 : l'orbite d'un etat donne
-- TODO etudiant : pour n = 2, decrire l'orbite complete de l'etat (true, false) :
-- Etape 1 : donner les etats successifs (pas 1, pas 2, pas 3)
-- Etape 2 : donner le point fixe atteint et le nombre de pas
-- Etape 3 : verifier avec l'oracle (reutiliser la table de la section 3)
#eval "Exercice 3 a completer"


-- Exercice 3 : l'orbite d'un etat donne
-- TODO etudiant : pour n = 2, decrire l'orbite complete de l'etat (true, false) :
-- Etape 1 : donner les etats successifs (pas 1, pas 2, pas 3)
-- Etape 2 : donner le point fixe atteint et le nombre de pas
-- Etape 3 : verifier avec l'oracle (reutiliser la table de la section 3)
#eval "Exercice 3 a completer"
─────▶  "Exercice 3 a completer"

--% env 15

Raw input:
{"cmd": "-- Exercice 3 : l'orbite d'un etat donne\n-- TODO etudiant : pour n = 2, decrire l'orbite complete de l'etat (true, false) :\n-- Etape 1 : donner les etats successifs (pas 1, pas 2, pas 3)\n-- Etape 2 : donner le point fixe atteint et le nombre de pas\n-- Etape 3 : verifier avec l'oracle (reutiliser la table de la section 3)\n#eval \"Exercice 3 a completer\"\n", "env": 14}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "\"Exercice 3 a completer\""}],
 "env": 15}

## Conclusion — vers le capstone

La marche 2 est en place : l'outil parité (4 lemmes), le système broadcast, le théorème B
(intégration universelle) et le théorème C (dynamique triviale), tous deux rendus par le
noyau et mesurés par l'oracle d'orbites. La marche 3 (capstone, `IIT-09`) assemblera le
contraste complet : côté Python, le proxy calculé sur des systèmes concrets ; côté Lean,
le couple B et C comme garde-fou formel — ce que le proxy récompense vs ce qu'il
devrait récompenser.

Ce que le lecteur emporte de ce carnet tient en trois acquis, et il vaut la peine de les nommer
avant de tourner la page.

1. **Un système de `2^n` états peut n'avoir qu'un bit d'information.** Le broadcast réduit un état à
   sa parité puis le diffuse ; son image a deux points, ses bassins sont ceux des classes de parité,
   et sa dynamique s'arrête en deux pas. Tout le reste en découle.
2. **La mesure et la preuve ne disent pas la même chose, et c'est leur écart qui instruit.**
   L'oracle énumère quatre ou huit états et montre le motif ; les théorèmes B et C le transportent à
   toute taille. Un carnet qui n'aurait que l'un des deux serait incomplet dans un sens précis :
   l'oracle seul laisse un angle mort sur les tailles non énumérées, les théorèmes seuls laissent le
   lecteur sans intuition.
3. **Un critère formel peut être exact et manquer son objet.** `integrated` est une définition
   vérifiée, et le broadcast la satisfait au maximum tout en ne calculant rien. C'est le genre de
   faille qu'une preuve rend visible et qu'un exemple seul laisserait discuter.

Le capstone reprendra l'objet côté Python — le calcul effectif du proxy sur des systèmes de taille
croissante — et le confrontera à ce que la marche 2 vient d'établir formellement. La question qu'il
posera n'est plus « le proxy vaut-il zéro ou un ? » mais « que mesure-t-il, au juste ? ».